In [2]:
import torch

In [353]:
# Linear

class Linear ( torch.nn.Module ):
    def __init__ ( self, fan_in , fan_out , device = None, dtype = None ):
        super().__init__()
        sigma = (2./ ( fan_in + fan_out ) ) **0.5
        self.w= torch.nn.Parameter ( torch.empty ( ( fan_in, fan_out ) , device = device, dtype = dtype ))
        torch.nn.init.trunc_normal_ ( self.w, std = sigma , a = -3. * sigma , b = + 3. * sigma ) 

    def forward ( self, xin ) :
        out =  xin @ self.w 
        return out

class Embedding ( torch.nn.Module ):
    def __init__ ( self, voc_size , dmodel , device = None, dtype = None ):
        super().__init__()
        self.w = torch.nn.Parameter  ( torch.empty ( ( voc_size , dmodel ) , device = device, dtype = dtype ))
    
        torch.nn.init.trunc_normal_ ( self.w, std = 1. , a = -3.  , b = + 3. ) 
    
    def forward ( self, xin ) :
        out = self.w[ xin ]
        return out

class RmsNorm ( torch.nn.Module ):
    def __init__ ( self, dmodel , eps = 1e-6, device = None, dtype = None ):
        super().__init__()
        self.gamma = torch.nn.Parameter ( torch.ones ( dmodel ,  device = device, dtype = dtype) ) 
        self.eps = eps
        self.dmoel = dmodel

    def forward (self, xin ) :
        
        sigma2 = torch.sum ( xin**2 , dim = -1, keepdim = True ) / self.dmoel
        out = xin * ( sigma2 + self.eps ) **-0.5 * self.gamma
        return out

def softmax ( xin, dim = -1) :
    xmax = torch.max ( xin, dim = dim, keepdim = True ).values 
    xexp =  (xin - xmax).exp()
    out = xexp / torch.sum ( xexp , dim = dim , keepdim = True )
    return out
    

def Attention ( qmat, kmat, vmat) :
    # why we need this function?
    T, dk = kmat.shape[-2:]


    out =  ( qmat @ torch.transpose ( kmat , -2, -1 ) ) * dk**-0.5 

    # should I worry that mask will attend the back propagation
    mask = torch.triu ( torch.ones( ( T, T ) , dtype = torch.bool ), diagonal = 1 )
    
    out =out.masked_fill ( mask, float('-inf')) 
    out = softmax ( out, dim = -1 )

    return out @ vmat
    

class RoPE (   torch.nn.Module ):
    def __init__ ( self, theta, d_k, max_length, device = None ):
        super().__init__()
        dk2 = d_k //2
        thetalist = torch.empty ( ( max_length, dk2 )) 
        for ii in range ( max_length ) :
            thetalist [ii]  = torch.tensor ([ ii * theta ** ( - jj / dk2 ) for jj in range (dk2) ])  

        #self.thetalist = thetalist
        sintheta = thetalist.sin()
        costheta = thetalist.cos()
        self.register_buffer ('sintheta', sintheta , persistent = False, )
        self.register_buffer ('costheta', costheta , persistent = False , )

    def forward ( self, xin, positions= None ) :
        out = torch.empty_like ( xin, device  = xin.device)
        x1 = xin [...,::2]
        x2 = xin [...,1::2]
        

        if positions is None :
            seqlen = xin.shape[-2]
            cth = self.costheta [:seqlen]
            sth = self.sintheta [:seqlen]
        else :
            cth = self.costheta [positions]
            sth = self.sintheta [positions]
            
        #print ( x1.shape, x2.shape, self.costheta.shape, self.sintheta.shape )
        x1new = x1 * cth - x2 * sth
        x2new  = +x1* sth + x2 * cth
        #print ( x1new.shape, x2new.shape)
        out [...,::2 ] = x1new
        out [...,1::2] = x2new
        return out

class MultiHeadAttention (  torch.nn.Module ):
    def __init__ ( self, dmodel , n_head, max_length, theta,  device = None, dtype = None ):
        super().__init__()
        dk = dmodel // n_head
        dv = dmodel // n_head
        self.Wq = Linear ( dmodel, dk * n_head , device = device, dtype = dtype )
        self.Wk = Linear ( dmodel, dk * n_head , device = device, dtype = dtype )
        self.Wv = Linear ( dmodel, dv * n_head , device = device, dtype = dtype )
        self.Wo = Linear ( dv * n_head, dmodel , device = device, dtype = dtype )
        self.n_head = n_head
        self.dk = dk
        self.dv = dv
        #print ( f"dim k  = {dk}")
        self.rope = RoPE ( theta, dk, max_length )  # why I want to instance Rope for every MHA?
        #print ( f" costheta shape , {self.rope.costheta.shape}")
        

    def forward ( self, xin, positions= None):
        # xin ( B, T, C )
        qmat = self.Wq ( xin )
        kmat = self.Wk ( xin )
        vmat = self.Wv ( xin ) 
        outlist = []
        for ii in range ( self.n_head ) :
            qi = qmat [...,ii * self.dk: (ii+1)*self.dk ]
            ki = kmat [...,ii * self.dk: (ii+1)*self.dk ]
            vi =  vmat [...,ii * self.dv: (ii+1)*self.dv ]
            #print ( qi.shape, ki.shape)
            
            qi = self.rope ( qi , positions = positions) 
            ki = self.rope ( ki, positions = positions )

            qkvi = Attention ( qi, ki, vi )
            outlist.append ( qkvi ) 
        
        out  = torch.cat ( outlist, dim = -1 )
        out = self.Wo ( out )
        return out
            
    

class FFN (  torch.nn.Module ):
    def __init__ ( self, dmodel ,dff,  device = None, dtype = None ):
        super().__init__()
        self.W1 =  Linear ( dmodel, dff , device = device, dtype = dtype )
        self.W2 =   Linear ( dmodel, dff , device = device, dtype = dtype )
        self.W3 =    Linear (  dff , dmodel, device = device, dtype = dtype )

    def forward ( self, xin ):
        x1 = self.W1 ( xin )
        x2  = self.W2 ( xin ) 
        out =  x1* torch.sigmoid ( x1) *   x2
        out = self.W3 ( out ) 
        return out

class transformer_block (  torch.nn.Module ):
    def __init__ ( self, dmodel ,n_head, dff, token_length, theta,  device = None, dtype = None ):
        super().__init__()
        self.ln1 = RmsNorm ( dmodel, device = device, dtype =dtype ) 
        self.mha = MultiHeadAttention ( dmodel , n_head, token_length, theta,  device = device, dtype = dtype )
        self.ln2 = RmsNorm ( dmodel, device = device, dtype =dtype ) 
        self.ffn = FFN ( dmodel ,dff , device = device, dtype =dtype ) 

    def forward ( self, xin ) :
        x = self.ln1 ( xin )
        x = xin  + self.mha ( x)
        y = self.ln2 ( x )
        y = x + self.ffn ( y )
        return y
        

class transformer_lm (  torch.nn.Module ):
    def __init__ ( self, voc_size, dmodel ,n_layers, n_head, dff, token_length, theta,  device = None, dtype = None ):
        super().__init__()

        self.emb = Embedding ( voc_size, dmodel,  device = device, dtype =dtype ) 
        self.blocks = torch.nn.ModuleList ( [ transformer_block (  dmodel ,n_head, dff, token_length, theta, 
                                                               device = device, dtype =dtype ) for _ in range ( n_layers ) ] )
        self.ln =  RmsNorm ( dmodel, device = device, dtype =dtype ) 
        self.linear = Linear ( dmodel, voc_size,  device = device, dtype =dtype ) 



    def forward ( self, xin  ) :
        x1 = self.emb ( xin ) 
        for mm in self.blocks :
            x1 = mm ( x1 )
        x1 = self.ln ( x1)
        x1 = self.linear ( x1) 
        #out = softmax ( x1, dim = -1 )
        return x1


def cross_entropy ( logits, ypred )  :
    logits = logits.view( -1, logits.shape[-1] )
    ypred = ypred.view ( -1)
    leny = len ( ypred )
    shifted = logits - logits.max ( dim =-1, keepdim = True).values
    #print ( l0.shape, y0.shape)
    #out = logits [ torch.arange ( leny ), ypred ] - shifted.exp().sum( dim = -1).log()
    out =shifted [ torch.arange ( leny ), ypred ] - shifted.exp().sum( dim = -1).log()
    
    return -out.mean()
    

        
        

In [247]:
## load data

import numpy as np

file_ids = "assignment1-basics/data/tinystories_valid.npy"
data = np.load ( file_ids, mmap_mode="r" )

In [252]:
len ( data)

5465883

**training**

In [355]:
Vocab_size = 10000
Context_length = 256
Dmodel = 512
Dff = round ( 8 * Dmodel / 3 / 64 ) * 64
Theta = 10000
Num_layers = 4
Num_heads = 16
Batch_size = 8

## initialize the transformer
tclass =  transformer_lm ( Vocab_size, Dmodel, Num_layers, Num_heads, Dff, Context_length, Theta )

lr = 1e-3
wdecay = 0.01
betas=(0.9, 0.999)
opt= torch.optim.AdamW(tclass.parameters(), lr=lr, weight_decay=wdecay)


In [356]:
Numrun = 100
lossi =[]
lendata = len ( data) 
xinput = torch.empty ( ( Batch_size, Context_length ), dtype = torch.long ) 
ypred = torch.empty ( ( Batch_size, Context_length ), dtype = torch.long ) 


for ii in range ( Numrun ) :
    opt.zero_grad ()
    blist = torch.randint ( lendata -Context_length  , (Batch_size,))
    for jj in range ( Batch_size ) :
        xinput [jj ] =  torch.tensor ( data [ blist [jj ] : blist [jj] + Context_length  ],)
        ypred [jj ] =  torch.tensor ( data [ blist [jj ] +1 : blist [jj] +1 + Context_length  ])

    logits = tclass ( xinput )
    #print ( logits.shape, ypred.shape)
    loss = cross_entropy ( logits, ypred ) 

    loss.backward()
    opt.step ()
    print ( f"step {ii}, loss = {loss.item() }")

    if ii == 10:
        break


    

step 0, loss = 9.264488220214844
step 1, loss = 8.569742202758789
step 2, loss = 7.874283790588379
step 3, loss = 7.375323295593262
step 4, loss = 6.960628509521484
step 5, loss = 6.479720592498779
step 6, loss = 6.225619316101074
step 7, loss = 5.864805698394775
step 8, loss = 5.7406182289123535
step 9, loss = 5.487415313720703
step 10, loss = 5.365272521972656


**test regime**

In [243]:
c1 = Linear ( 10, 20)
c1.w.shape
a = torch.ones ( ( 20, 10))
a = c1(a)

V = 100
dmodel = 16
n_head = 2
n_layers = 4
max_length = 8
theta =1000
batch_size= 2
dff = int ( (  8/3 * dmodel // 4) *4 )

c2 = Embedding ( V, dmodel ) 
toks = torch.ones ( (5,6), dtype = torch.long)
c2( toks).shape

c3= RmsNorm ( dmodel)
b=c3 ( a)
b.shape

b1=softmax ( b, dim =-1)
torch.sum ( b1, dim =-1)
a = torch.ones ( ( 2, 3, 4 ))
b = torch.ones ( ( 2, 3, 4 ))
c = torch.ones ( ( 2, 3 , 4) )
d =Attention ( a, b, c )
d.shape

class1= RoPE ( 1000,  4,6)
a = torch.ones ( ( 2, 6, 4 ))
b = torch.ones ( ( 2, 6, 4 ))

class2 = MultiHeadAttention ( dmodel, n_head, max_length, theta =theta )
x = torch.rand ( ( batch_size, max_length , dmodel ))
xout =class2 ( x )

class3 = FFN ( dmodel, dff )
class3 ( xout ).shape

class4 =  transformer_block ( dmodel ,n_head, dff, max_length, theta, )
x = torch.rand ( ( batch_size, max_length , dmodel ))
class4 ( x)

class5 = transformer_lm ( V, dmodel ,n_layers, n_head, dff, max_length, theta, )  
toks = torch.ones ( (batch_size,max_length), dtype = torch.long)
class5(toks).shape

torch.Size([2, 8, 100])

In [134]:
a= torch.randn ( ( 2,3))
a [...,1::2].shape, a.shape

(torch.Size([2, 1]), torch.Size([2, 3]))

In [105]:
a [...,::2]

tensor([[ 1.0170,  0.1815],
        [-0.5750,  0.6498]])

In [113]:
a.shape
b = torch.ones ( (2, 3))
a*b

RuntimeError: The size of tensor a (4) must match the size of tensor b (2) at non-singleton dimension 1

In [122]:
a.shape

torch.Size([2, 6, 4])

In [139]:
a [:2]

tensor([[-0.0376,  1.7922, -0.2591],
        [-1.0459,  1.1961, -0.6798]])

In [137]:
a

tensor([[-0.0376,  1.7922, -0.2591],
        [-1.0459,  1.1961, -0.6798]])

In [147]:
a.shape

torch.Size([2, 6, 4])

In [156]:
a = torch.randn(3,2, 3)
torch.cat([a,a,a],-1).shape

torch.Size([3, 2, 9])

In [219]:
a[1,1,-2:]

tensor([1., 1.])

In [240]:
a =torch.ones ( (3,3), dtype = torch.bool)

In [241]:
b= torch.triu( a, diagonal =1 )
b

tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])

In [237]:
torch.ones_like (a)

tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]]])

In [255]:
a =torch.randint ( 4, (5,))

tensor([2, 3, 0, 1, 2])

In [283]:
a.mean()

tensor(1.)

In [315]:
a= torch.randn ( (4,5,3) )

In [298]:
a 

tensor([[[-0.7877,  0.9267, -1.2693, -0.0776],
         [-1.3012, -1.3249,  0.3965, -1.8307],
         [-1.1776,  0.5552,  1.4346, -0.6584]],

        [[ 0.6447,  1.2244,  0.5666,  0.8990],
         [-0.0607, -1.6085, -1.2505,  0.5198],
         [ 1.1612,  1.0234,  0.3987,  0.9808]],

        [[-0.4875,  1.5799, -0.1960, -0.0513],
         [-0.3382, -1.4695, -0.1468, -1.3716],
         [ 1.4101,  0.1955,  0.2965,  2.4298]],

        [[-0.0783, -0.5836, -0.2631,  2.5556],
         [ 0.2220,  0.7774,  2.5153, -0.8113],
         [-0.6041,  0.3312,  0.5538, -0.0894]]])

In [314]:
b =torch.randint ( 3, (4,5))
b

tensor([[1, 0, 2, 1, 2],
        [0, 0, 1, 2, 2],
        [2, 0, 0, 0, 1],
        [0, 0, 2, 2, 0]])

In [304]:
a[b].shape

torch.Size([4, 3])

In [317]:
a[torch.arange(b.shape[0]), torch.arange (b.shape[1]),b]

IndexError: shape mismatch: indexing tensors could not be broadcast together with shapes [4], [5], [4, 5]

In [313]:
a,b

(tensor([[-1.1734e+00, -5.8159e-01,  1.5029e-01],
         [-1.4588e+00, -3.0673e-01, -5.8701e-01],
         [-5.5147e-01,  1.2054e+00, -1.1839e-03],
         [-2.3407e-01,  4.3865e-01, -4.5218e-01]]),
 tensor([1, 1, 1, 0]))

In [319]:
torch.arange(b.shape[1])

tensor([0, 1, 2, 3, 4])

In [320]:
a.view( -1, a.shape[-1] )

tensor([[ 1.5694, -0.5172,  0.7964],
        [ 0.8804, -0.8788, -1.1447],
        [-0.4739,  1.1134, -0.8812],
        [ 0.8884,  0.2788,  1.4906],
        [-0.1596,  0.4913, -0.6182],
        [ 1.2684, -1.1454,  2.5855],
        [-0.3158,  1.1422,  0.0679],
        [ 0.3250,  0.1046,  0.6418],
        [-1.1003, -1.8257,  1.3074],
        [-0.5102,  0.3625, -1.4121],
        [-0.9537, -0.4528, -0.4793],
        [ 0.2912, -1.0672, -0.8120],
        [ 0.2621,  1.1920,  1.3696],
        [-0.1065,  2.4032, -0.8717],
        [ 0.3381, -0.4609, -0.7607],
        [ 0.4492, -0.8916, -0.4959],
        [ 1.6395,  0.9529, -1.4645],
        [ 1.4650, -0.8338,  0.9466],
        [ 0.7575,  1.5252, -0.8373],
        [ 0.5661, -0.8251, -1.4221]])

In [321]:
a

tensor([[[ 1.5694, -0.5172,  0.7964],
         [ 0.8804, -0.8788, -1.1447],
         [-0.4739,  1.1134, -0.8812],
         [ 0.8884,  0.2788,  1.4906],
         [-0.1596,  0.4913, -0.6182]],

        [[ 1.2684, -1.1454,  2.5855],
         [-0.3158,  1.1422,  0.0679],
         [ 0.3250,  0.1046,  0.6418],
         [-1.1003, -1.8257,  1.3074],
         [-0.5102,  0.3625, -1.4121]],

        [[-0.9537, -0.4528, -0.4793],
         [ 0.2912, -1.0672, -0.8120],
         [ 0.2621,  1.1920,  1.3696],
         [-0.1065,  2.4032, -0.8717],
         [ 0.3381, -0.4609, -0.7607]],

        [[ 0.4492, -0.8916, -0.4959],
         [ 1.6395,  0.9529, -1.4645],
         [ 1.4650, -0.8338,  0.9466],
         [ 0.7575,  1.5252, -0.8373],
         [ 0.5661, -0.8251, -1.4221]]])